# NB20 보안 예산 — 20장

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB20-1: 비트코인 보조금 일정 — Core 의 GetBlockSubsidy 를 정수 산술 그대로 옮긴다 (공부용)
COIN = 100_000_000                      # 1 BTC = 1e8 사토시 (consensus/amount.h)
HALVING = 210_000                       # nSubsidyHalvingInterval (chainparams.cpp)

def block_subsidy_sat(height: int) -> int:
    halvings = height // HALVING
    if halvings >= 64:                  # validation.cpp: 64번 이상 반감하면 0 (비트 이동 폭 보호)
        return 0
    return (50 * COIN) >> halvings      # 50 BTC 를 halvings 번 반으로 — 정수 비트 이동이라 1 사토시 아래는 버려진다

print(f"{'반감 횟수':>6} {'시작 높이':>10} {'블록 보조금(BTC)':>18} {'그 구간 끝 누적 발행(BTC)':>24}")
cum = 0
for h in range(0, 36):
    start = h * HALVING
    sub = block_subsidy_sat(start)
    cum += sub * HALVING
    if h <= 6 or sub == 0 or h in (10, 20, 32, 33, 34):
        print(f"{h:>6} {start:>10,} {sub / COIN:>18.8f} {cum / COIN:>24.8f}")
    if sub == 0:
        break
total = sum(block_subsidy_sat(h * HALVING) * HALVING for h in range(0, 64))
print(f"\n정수 산술로 낸 보조금 총합: {total:,} 사토시 = {total / COIN:.8f} BTC (2,100만에 {21_000_000 - total / COIN:.8f} BTC 모자란다)")
tip = 972_958                           # 2026-10-24 UTC 관측 체인 끝 높이 (data/hashrate_2026-10-25.json)
print(f"기준일 높이 {tip:,}: 반감 {tip // HALVING}회 뒤, 블록 보조금 {block_subsidy_sat(tip) / COIN} BTC, 다음 반감 높이 {(tip // HALVING + 1) * HALVING:,} (약 {((tip // HALVING + 1) * HALVING - tip) * 10 / 60 / 24:.0f}일 뒤, 블록당 10분 가정)")

 반감 횟수      시작 높이        블록 보조금(BTC)        그 구간 끝 누적 발행(BTC)
     0          0        50.00000000        10500000.00000000
     1    210,000        25.00000000        15750000.00000000
     2    420,000        12.50000000        18375000.00000000
     3    630,000         6.25000000        19687500.00000000
     4    840,000         3.12500000        20343750.00000000
     5  1,050,000         1.56250000        20671875.00000000
     6  1,260,000         0.78125000        20835937.50000000
    10  2,100,000         0.04882812        20989746.09270000
    20  4,200,000         0.00004768        20999989.97250000
    32  6,720,000         0.00000001        20999999.97690000
    33  6,930,000         0.00000000        20999999.97690000

정수 산술로 낸 보조금 총합: 2,099,999,997,690,000 사토시 = 20999999.97690000 BTC (2,100만에 0.02310000 BTC 모자란다)
기준일 높이 972,958: 반감 4회 뒤, 블록 보조금 3.125 BTC, 다음 반감 높이 1,050,000 (약 535일 뒤, 블록당 10분 가정)


In [ ]:
# NB20-2: 부디시 명제 3 — 이중지불 공격의 기대 지속시간 t(A, e, k) 와 총비용 A·t 를 정직한 블록 보상 단위로 (공부용)
# 계산에 들어가는 것은 공격자 배수 A(점유율 A/(1+A))와 블록 수 k+e 뿐이다. 코인 가격·법정통화 단가는 없다.
from math import lgamma, log, exp

def attack_duration(A: float, ke: int) -> float:
    """식 (6). 계승은 로그감마로 — 큰 ke 에서 넘침을 피한다. 단위: 정직한 블록 시간(≈10분)."""
    t = float(ke)
    for i in range(0, ke + 1):
        log_comb = lgamma(2 * ke - i) - lgamma(ke - i + 1) - lgamma(ke)   # (2ke−1−i)! / ((ke−i)! (ke−1)!)
        log_prob = (ke - i) * log(A / (1 + A)) + ke * log(1 / (1 + A))
        t += (i + 1) / (A - 1) * exp(log_comb + log_prob)
    return t

rows = [1.05, 1.10, 1.20, 1.33, 1.50, 2.00, 5.00]
cols = [1, 6, 12, 36, 144]
print("기대 지속시간 t (단위: 정직한 블록 시간). 열 = k+e (이중지불 블록 수 + 에스크로)")
print(f"{'A (점유율)':>14}" + "".join(f"{c:>10}" for c in cols))
for A in rows:
    print(f"{A:>5.2f} ({A/(1+A):4.0%})" + "".join(f"{attack_duration(A, c):>10.2f}" for c in cols))
print("\n총비용 A·t (단위: 블록당 정직한 신뢰지원 비용 N*c = 블록 보상 p_block)")
print(f"{'A (점유율)':>14}" + "".join(f"{c:>10}" for c in cols))
for A in rows:
    print(f"{A:>5.2f} ({A/(1+A):4.0%})" + "".join(f"{A * attack_duration(A, c):>10.2f}" for c in cols))

# 공격자는 A·t 를 최소로 하는 A 를 고른다. k+e = 12(논문의 기본 사례)에서 격자 탐색
ke = 12
best = min(((A / 100) * attack_duration(A / 100, ke), A / 100) for A in range(101, 600))
print(f"\nk+e = {ke}: A·t 를 최소로 하는 A* ≈ {best[1]:.2f} (점유율 {best[1]/(1+best[1]):.0%}), 최소 A*·t(A*) ≈ {best[0]:.2f} 블록 보상")
print(f"→ 식 (3): p_block > V_attack / {best[0]:.1f}, 곧 블록당 보상이 공격 가치의 {1/best[0]:.2%} 를 넘어야 한다")

기대 지속시간 t (단위: 정직한 블록 시간). 열 = k+e (이중지불 블록 수 + 에스크로)
       A (점유율)         1         6        12        36       144
 1.05 ( 51%)     25.51     42.43     56.59     97.91    229.61
 1.10 ( 52%)     13.02     22.88     31.73     60.14    167.31
 1.20 ( 55%)      6.79     13.24     19.69     43.10    146.79
 1.33 ( 57%)      4.34      9.60     15.35     38.04    144.18
 1.50 ( 60%)      3.08      7.84     13.43     36.47    144.00
 2.00 ( 67%)      1.89      6.45     12.20     36.01    144.00
 5.00 ( 83%)      1.12      6.00     12.00     36.00    144.00

총비용 A·t (단위: 블록당 정직한 신뢰지원 비용 N*c = 블록 보상 p_block)
       A (점유율)         1         6        12        36       144
 1.05 ( 51%)     26.78     44.56     59.42    102.81    241.09
 1.10 ( 52%)     14.32     25.16     34.90     66.16    184.04
 1.20 ( 55%)      8.14     15.89     23.63     51.72    176.15
 1.33 ( 57%)      5.78     12.77     20.41     50.59    191.76
 1.50 ( 60%)      4.62     11.76     20.14     54.71    216.01
 2.00 ( 6

In [ ]:
# NB20-3: 공격 비용 계산기 — 입력은 해시율(스냅샷)·공격자 배수·단가·기간. 코인 가격은 입력이 아니다 (공부용)
import json, os

# 기준일 해시율 스냅샷(데이터 동봉; 파일이 없으면 같은 값의 상수로 대체 — Colab 에서 파일 없이 실행 가능)
try:
    with open(os.path.join("data", "hashrate_2026-10-25.json"), encoding="utf-8") as f:
        snap = json.load(f)
    H_network = float(snap["estimate"]["hashrate_H_per_s"])
except FileNotFoundError:
    H_network = 1.006294e21                 # H/s — 난이도 140,577,680,051,919.6 에서 D·2^32/600 으로 추정(600초 가정)
print(f"기준일 추정 해시율 H = {H_network:.3e} H/s = {H_network/1e18:,.0f} EH/s (2026-10-24 UTC, 600초 가정 추정 — 비교값보다 약 7% 작다)")

def attack_cost(A, ke, c_per_hash_second=None, p_block=None):
    """총비용. c_per_hash_second 를 주면 「단가 단위 × 해시·초」, p_block 을 주면 식 (1)로 「블록 보상 단위」."""
    t_blocks = attack_duration(A, ke)
    seconds = t_blocks * 600
    out = {"t_blocks": t_blocks, "hours": seconds / 3600, "attacker_hashrate_hs": A * H_network}
    if c_per_hash_second is not None:
        out["cost_in_price_units"] = A * H_network * c_per_hash_second * seconds
    if p_block is not None:
        out["cost_in_block_rewards"] = A * t_blocks            # 식 (1): N*c = p_block
        out["cost_in_coin_of_reward"] = A * t_blocks * p_block
    return out

p_block_subsidy_only = 3.125                 # 기준일 보조금(BTC). 수수료 몫 F 는 블록마다 다르므로 여기서는 0 으로 둔 하한
print(f"\n(가) 블록 보상 단위 — 수수료를 0 으로 둔 하한, p_block = {p_block_subsidy_only} BTC(보조금만)")
print(f"{'점유율':>6} {'A':>5} {'k+e':>4} {'기대 블록':>9} {'시간(h)':>8} {'블록 보상 개수':>12} {'BTC 상당(보조금만)':>18}")
for q in [0.55, 0.60, 0.67]:
    A = q / (1 - q)
    for ke in [6, 12, 144]:
        r = attack_cost(A, ke, p_block=p_block_subsidy_only)
        print(f"{q:>6.0%} {A:>5.2f} {ke:>4} {r['t_blocks']:>9.1f} {r['hours']:>8.1f} {r['cost_in_block_rewards']:>12.1f} {r['cost_in_coin_of_reward']:>18.1f}")

print("\n(나) 사용자 단가 단위 — c 는 해시 1회당 비용이 아니라 「1 H/s 를 1초 돌리는 비용」. 아래 값은 자리만 채운 가상의 단가다.")
c_demo = 1e-19                                # 가상 단가(단위 없음). 실제 전력·장비 단가는 사용자가 넣는다
for q in [0.55, 0.60]:
    A = q / (1 - q)
    r = attack_cost(A, 12, c_per_hash_second=c_demo)
    print(f"  점유율 {q:.0%}, k+e=12: 공격자 해시율 {r['attacker_hashrate_hs']:.2e} H/s × {r['hours']:.1f}시간 × 단가 → 비용 {r['cost_in_price_units']:,.0f} (단가 단위)")
print("\n같은 점유율·같은 기간에서 비용은 단가 c 에 비례한다. 단가를 반으로 넣으면 비용도 반 — 이 계산기가 말하는 것은 수준이 아니라 비례다.")

기준일 추정 해시율 H = 1.006e+21 H/s = 1,006 EH/s (2026-10-24 UTC, 600초 가정 추정 — 비교값보다 약 7% 작다)

(가) 블록 보상 단위 — 수수료를 0 으로 둔 하한, p_block = 3.125 BTC(보조금만)
   점유율     A  k+e     기대 블록    시간(h)     블록 보상 개수       BTC 상당(보조금만)
   55%  1.22    6      12.3      2.0         15.0               47.0
   55%  1.22   12      18.5      3.1         22.7               70.8
   55%  1.22  144     145.8     24.3        178.2              556.7
   60%  1.50    6       7.8      1.3         11.8               36.7
   60%  1.50   12      13.4      2.2         20.1               62.9
   60%  1.50  144     144.0     24.0        216.0              675.0
   67%  2.03    6       6.4      1.1         13.0               40.7
   67%  2.03   12      12.2      2.0         24.7               77.3
   67%  2.03  144     144.0     24.0        292.4              913.6

(나) 사용자 단가 단위 — c 는 해시 1회당 비용이 아니라 「1 H/s 를 1초 돌리는 비용」. 아래 값은 자리만 채운 가상의 단가다.
  점유율 55%, k+e=12: 공격자 해시율 1.23e+21 H/s × 3.1시간 × 단가 → 비용 1,368,343 (단가 단위)
  점유율 60%,